# Окутин Денис ИУ9 - 21М
**Домашняя работа №4**

In [1]:
import math
import os
from collections import Counter
import pymorphy3
from razdel import tokenize, sentenize
import pdfplumber

morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):
    tokens = list(tokenize(text))
    lemmas = []
    for tok in tokens:
        if tok.text.isalpha():
            parsed = morph.parse(tok.text)[0]
            lemmas.append(parsed.normal_form)
    return lemmas

def extract_sentences_from_pdf(pdf_path):
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                all_text += page_text + "\n"
    
    raw_sents = [s.text for s in sentenize(all_text) if s.text.strip()]
    cleaned_sents = [' '.join(sent.split()) for sent in raw_sents]
    
    return cleaned_sents

In [41]:
pages_dir = "../pages"
all_sentences = []
all_lemma_sentences = []

for filename in os.listdir(pages_dir):
    if filename.lower().endswith('.pdf'):
        pdf_path = os.path.join(pages_dir, filename)
        try:
            sents = extract_sentences_from_pdf(pdf_path)
            print(f"Файл {filename}: извлечено {len(sents)} предложений.")
            all_sentences.extend(sents)
            for sent in sents:
                all_lemma_sentences.append(preprocess_text(sent))
        except Exception as e:
            print(f"Ошибка при обработке {filename}: {e}")

print(f"\nВсего собрано предложений: {len(all_sentences)}")

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Файл Лёгкая_атлетика.pdf: извлечено 174 предложений.


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Файл Чикаго.pdf: извлечено 317 предложений.


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Файл Монгольфьер.pdf: извлечено 88 предложений.
Файл Ла-Верна.pdf: извлечено 113 предложений.


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Файл Нью-Йорк.pdf: извлечено 587 предложений.


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Файл Котанджа,_Мелих.pdf: извлечено 19 предложений.
Файл Нью-Йорк_сауэр.pdf: извлечено 32 предложений.

Всего собрано предложений: 1330


In [33]:
all_lemma_sentences[:3]

[['лёгкий',
  'атлетика',
  'материал',
  'из',
  'википедия',
  'свободный',
  'энциклопедия',
  'лёгкий',
  'олимпийский',
  'вид',
  'спорт'],
 ['лёгкий',
  'атлетика',
  'объединять',
  'следующий',
  'дисциплина',
  'беговой',
  'вид',
  'спортивный',
  'ходьба',
  'технический',
  'вид',
  'прыжок',
  'и',
  'метание',
  'многоборье',
  'пробег',
  'бег',
  'по',
  'шоссе',
  'и',
  'кросс',
  'бег',
  'по',
  'пересечь',
  'местность'],
 ['один', 'из', 'основный', 'и', 'наиболее', 'массовый', 'вид', 'спорт']]

In [34]:
N = len(all_lemma_sentences)

df = Counter()
for sent in all_lemma_sentences:
    unique_words = set(sent)
    for word in unique_words:
        df[word] += 1
        
    
idf = {}
for word, freq in df.items():
    idf[word] = math.log(N/freq)
    

doc_lengths = [len(sent) for sent in all_lemma_sentences]
avgdl = sum(doc_lengths) / N

doc_tfs = [Counter(sent) for sent in all_lemma_sentences]

In [35]:
def compute_norm(tf_dict, idf=None):
    norm = 0.0
    if idf is None:
        for w in tf_dict.values():
            norm += w * w
    else:
        for word, tf in tf_dict.items():
            w = tf * idf.get(word, 0)
            norm += w * w
    return math.sqrt(norm)

def cosine_similarity(vec_q, vec_d, norm_q, norm_d):
    dot = 0.0
    for word, wq in vec_q.items():
        if word in vec_d:
            dot += wq * vec_d[word]
    if norm_q == 0 or norm_d == 0:
        return 0.0
    return dot / (norm_q * norm_d)

K1 = 1.2
B = 0.75

def bm25_score(query_tf, doc_tf, doc_len, avgdl, idf):
    score = 0.0
    K = K1 * (1 - B + B * doc_len / avgdl)
    for word, tf_q in query_tf.items():
        if word in doc_tf:
            tf_d = doc_tf[word]
            numerator = tf_d * (K1 + 1)
            denominator = tf_d + K
            score += idf.get(word, 0) * numerator / denominator
    return score

In [36]:
norm_tf_docs = [compute_norm(tf) for tf in doc_tfs]
norm_tfidf_docs = [compute_norm(tf, idf) for tf in doc_tfs]

In [37]:
queries = [
    "Турецкий футболист успешно выступал и в лёгкой атлетике.",
    "Коктейль назвали в честь Нью-Йорка, хотя, вероятно, создан он был в Чикаго.",
    "В гроте карстовой пещеры инженеры-студенты летали на воздушном шаре, а инженеры-профессионалы — запустили электростанцию."
]

for q_idx, query_text in enumerate(queries, 1):
    print(f"Запрос {q_idx}: {query_text}")
    
    query_lemmas = preprocess_text(query_text)
    query_tf = Counter(query_lemmas)
    print(f"Леммы запроса: {query_lemmas}")
    print(f"TF запроса: {dict(query_tf)}")
    
    norm_tf_q = compute_norm(query_tf)
    query_tfidf = {word: tf * idf.get(word, 0) for word, tf in query_tf.items()}
    norm_tfidf_q = compute_norm(query_tf, idf)
    
    results_tf = []
    results_tfidf = []
    results_bm25 = []
    
    for i in range(N):
        sim_tf = cosine_similarity(query_tf, doc_tfs[i], norm_tf_q, norm_tf_docs[i])
        results_tf.append((sim_tf, all_sentences[i]))
        
        doc_tfidf = {word: tf * idf.get(word, 0) for word, tf in doc_tfs[i].items()}
        sim_tfidf = cosine_similarity(query_tfidf, doc_tfidf, norm_tfidf_q, norm_tfidf_docs[i])
        results_tfidf.append((sim_tfidf, all_sentences[i]))
        
        bm25 = bm25_score(query_tf, doc_tfs[i], doc_lengths[i], avgdl, idf)
        results_bm25.append((bm25, all_sentences[i]))
    
    results_tf.sort(key=lambda x: x[0], reverse=True)
    results_tfidf.sort(key=lambda x: x[0], reverse=True)
    results_bm25.sort(key=lambda x: x[0], reverse=True)
    
    print("\n--- Модель TF (косинусная мера) ---")
    for score, sent in results_tf[:10]:
        print(f"{score:.6f}\t{sent}")
    
    print("\n--- Модель TF-IDF (косинусная мера) ---")
    for score, sent in results_tfidf[:10]:
        print(f"{score:.6f}\t{sent}")
    
    print("\n--- Модель BM25 ---")
    for score, sent in results_bm25[:10]:
        print(f"{score:.6f}\t{sent}")
    print()

Запрос 1: Турецкий футболист успешно выступал и в лёгкой атлетике.
Леммы запроса: ['турецкий', 'футболист', 'успешно', 'выступать', 'и', 'в', 'лёгкий', 'атлетика']
TF запроса: {'турецкий': 1, 'футболист': 1, 'успешно': 1, 'выступать': 1, 'и': 1, 'в': 1, 'лёгкий': 1, 'атлетика': 1}

--- Модель TF (косинусная мера) ---
0.516398	Карьера в лёгкой атлетике Помимо футбола, Котанджа выступал в лёгкой атлетике, специализируясь на беге на различные [2] дистанции, пятиборье, многоборье и метании копья .
0.441942	Он выступал в лёгкой атлетике, специализируясь на беге на различные дистанции, пятиборье и многоборье.
0.408248	В связи с этим различаются два сезона лёгкой атлетики, в регионах, где эта спортивная дисциплина наиболее популярна — в Европе и США.
0.392232	И сегодня Олимпийские игры — мощный стимул для развития лёгкой атлетики во всем мире.
0.385758	Современная лёгкая атлетика начала свой путь с отдельных попыток в разных странах проводить соревнования в беге, прыжках и метаниях.
0.385758	

# Отчёт

## 1. Запросы и ожидаемые результаты

**Запрос 1:**  
*Турецкий футболист успешно выступал и в лёгкой атлетике.*  
Ожидаемые предложения (согласно ручному анализу):  
- «Мели́х Котанджа́ … — турецкий футболист, играл на позиции нападающего.»  
- «Он выступал в лёгкой атлетике, специализируясь на беге на различные дистанции, пятиборье и многоборье.»

**Запрос 2:**  
*Коктейль назвали в честь Нью-Йорка, хотя, вероятно, создан он был в Чикаго.*  
Ожидаемое предложение:  
- «Предполагается, что Нью-Йорк сауэр был изобретён в 1880‑е годы в Чикаго как вариация виски сауэра, впервые упомянутого в 1862 году.»

**Запрос 3:**  
*В гроте карстовой пещеры инженеры-студенты летали на воздушном шаре, а инженеры-профессионалы — запустили электростанцию.*  
Ожидаемые фрагменты (разбросаны по статье):  
- упоминание грота Ла-Верна в карстовой пещере;  
- полёт студентов на воздушном шаре под землёй;  
- запуск гидроэлектростанции в 2008 году.

## 2. Результаты и их анализ

### 3.1. Запрос 1

**Лучшие предложения по трём моделям (первые 5):**

| Модель | Предложение | Вес |
|--------|-------------|-----|
| **TF** | Карьера в лёгкой атлетике Помимо футбола, Котанджа выступал в лёгкой атлетике, специализируясь на беге на различные [2] дистанции, пятиборье, многоборье и метании копья . | 0.516 |
| | Он выступал в лёгкой атлетике, специализируясь на беге на различные дистанции, пятиборье и многоборье. | 0.442 |
| **TF‑IDF** | Мелих Котанджа 1915 — 8 июня 1986, Стамбул) — турецкий футболист, играл на позиции нападающего. | 0.397 |
| | Он выступал в лёгкой атлетике, специализируясь на беге на различные дистанции, пятиборье и многоборье. | 0.298 |
| **BM25** | Мелих Котанджа 1915 — 8 июня 1986, Стамбул) — турецкий футболист, играл на позиции нападающего. | 15.012 |
| | Карьера в лёгкой атлетике Помимо футбола, Котанджа выступал в лёгкой атлетике... | 13.939 |

Все три модели успешно находят ключевые предложения, подтверждающие факт. Модель TF отдаёт предпочтение предложению, где много слов из запроса («выступал», «лёгкая атлетика»), но не включает явное упоминание «турецкий футболист». TF‑IDF и BM25, благодаря весам IDF, выше оценивают редкое слово «Котанджа» и сочетание «турецкий футболист», поэтому предложение с именем футболиста выходит на первое место. Таким образом, для первого запроса все модели справились хорошо, причём TF‑IDF и BM25 точнее отражают специфику запроса.

### 3.2. Запрос 2

**Лучшие предложения по трём моделям:**

| Модель | Предложение | Вес |
|--------|-------------|-----|
| **TF** | В астрономии В честь Чикаго назван астероид (334) Чикаго, открытый в 1892 году. | 0.636 |
| | 19 сентября 1783 года в Версале... воздушный шар взмыл... | 0.471 |
| **TF‑IDF** | В астрономии В честь Чикаго назван астероид (334) Чикаго... | 0.345 |
| | [1] Коктейль похож на виски сауэр , отличается от него добавлением красного вина. | 0.161 |
| **BM25** | В астрономии В честь Чикаго назван астероид (334) Чикаго... | 16.110 |
| | [1] Коктейль похож на виски сауэр , отличается от него добавлением красного вина. | 9.635 |

Ожидаемое предложение об изобретении коктейля в Чикаго не попало в топ‑5 ни одной модели. Вероятные причины:
- Запрос содержит синоним «создан», тогда как в статье используется «изобретён» – лемматизация приводит оба слова к общему виду («создать» и «изобрести» остаются разными леммами), поэтому лексическое совпадение отсутствует.
- Информация о названии в честь Нью‑Йорка и о создании в Чикаго находится в разных частях статьи, и ни одно предложение не объединяет все ключевые понятия («коктейль», «Нью‑Йорк», «Чикаго», «назвать», «создать»).
- Модели находят либо предложения с «Чикаго» и «в честь» (астероид), либо предложения с «коктейль» (похож на виски сауэр), но не их комбинацию.

### 3.3. Запрос 3

**Лучшие предложения по трём моделям:**

| Модель | Предложение | Вес |
|--------|-------------|-----|
| **TF** | Полёт на воздушном шаре В декабре 2001 года студенты парижской Политехнической школы задумали уникальный проект: полёт на воздушном шаре под землёй. | 0.399 |
| | Галерея Монгольфьер ... (длинный фрагмент с перечислением) | 0.339 |
| **TF‑IDF** | Полёт на воздушном шаре В декабре 2001 года студенты... | 0.223 |
| | По состоянию на 2020 год Пьер-Сен-Мартен, [1] насчитывающая около 2 тысяч гротов и имеющая нижнюю отметку в —1342 м , входит в топ‑30 пещер... | 0.186 |
| | Регион Новая Аквитания ... История Первые экспедиции Открытие Ла-Верны Сооружение туннеля Полёт на воздушном шаре Запуск электростанции ... | 0.157 |
| **BM25** | Полёт на воздушном шаре В декабре 2001 года студенты... | 13.682 |
| | Регион Новая Аквитания ... Полёт на воздушном шаре Запуск электростанции ... | 11.188 |
| | Ла-Верна Материал из Википедии ... Зал Ла-Верна … гигантский грот … в карстовой пещере Гуфр-де-Партаж ... | 10.846 |
 
Здесь модели продемонстрировали впечатляющую способность собирать разрозненные фрагменты. В топе присутствуют:
- предложение о полёте студентов на воздушном шаре (точно соответствует части факта);
- фрагменты, описывающие грот Ла-Верна и карстовую пещеру;
- в одном из предложений (из оглавления или списка разделов) перечислены «Полёт на воздушном шаре» и «Запуск электростанции», что даёт высокий балл, особенно у BM25, благодаря наличию всех ключевых слов (кроме «грот», но есть «пещера»).

Модель TF тоже находит предложение о полёте, но второе место занимает «Галерея Монгольфьер» — длинный фрагмент, где много раз повторяется «монгольфьер», что повышает TF, но фактической информации о гроте и электростанции там нет.

BM25 и TF‑IDF, напротив, выделяют редкие термины («грот», «карстовый», «электростанция»), поэтому в топ попадают содержательные предложения. Особенно удачно сработал BM25, который на втором месте выдал предложение, объединяющее и полёт, и запуск электростанции (пусть и в виде заголовков разделов).

## 4. Сравнение моделей

- **TF:** хорошо находит предложения, содержащие много слов из запроса, но часто выдаёт «шумные» длинные фрагменты с повторяющимися словами (например, галерея монгольфьеров). Не учитывает редкость терминов.
- **TF‑IDF:** улучшает результаты, придавая больший вес специфическим словам. Для первого и третьего запросов это позволило поднять релевантные предложения выше. Для второго запроса TF‑IDF всё же не смог преодолеть синонимический барьер.
- **BM25:** в целом показал себя лучше всех – сочетание учёта IDF, длины документа и насыщения частоты дало наиболее релевантные результаты. В первом запросе он вывел на первое место именно предложение с именем футболиста, в третьем – собрал разрозненные факты.

**Домашняя работа №9**

In [8]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [57]:
from gensim.models import FastText

FASTTEXT_PATH = "cc.ru.300.bin"

print("Загрузка fastText модели...")
model = FastText.load_fasttext_format("cc.ru.300.bin")
print("Модель загружена. Размерность:", model.vector_size)

Загрузка fastText модели...
Модель загружена. Размерность: 300


In [58]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')
from nltk.corpus import stopwords
russian_stopwords = set(stopwords.words('russian'))
print(f"Загружено {len(russian_stopwords)} стоп-слов")
russian_stopwords

Загружено 151 стоп-слов


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/d.okutin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


{'а',
 'без',
 'более',
 'больше',
 'будет',
 'будто',
 'бы',
 'был',
 'была',
 'были',
 'было',
 'быть',
 'в',
 'вам',
 'вас',
 'вдруг',
 'ведь',
 'во',
 'вот',
 'впрочем',
 'все',
 'всегда',
 'всего',
 'всех',
 'всю',
 'вы',
 'где',
 'да',
 'даже',
 'два',
 'для',
 'до',
 'другой',
 'его',
 'ее',
 'ей',
 'ему',
 'если',
 'есть',
 'еще',
 'ж',
 'же',
 'за',
 'зачем',
 'здесь',
 'и',
 'из',
 'или',
 'им',
 'иногда',
 'их',
 'к',
 'как',
 'какая',
 'какой',
 'когда',
 'конечно',
 'кто',
 'куда',
 'ли',
 'лучше',
 'между',
 'меня',
 'мне',
 'много',
 'может',
 'можно',
 'мой',
 'моя',
 'мы',
 'на',
 'над',
 'надо',
 'наконец',
 'нас',
 'не',
 'него',
 'нее',
 'ней',
 'нельзя',
 'нет',
 'ни',
 'нибудь',
 'никогда',
 'ним',
 'них',
 'ничего',
 'но',
 'ну',
 'о',
 'об',
 'один',
 'он',
 'она',
 'они',
 'опять',
 'от',
 'перед',
 'по',
 'под',
 'после',
 'потом',
 'потому',
 'почти',
 'при',
 'про',
 'раз',
 'разве',
 'с',
 'сам',
 'свою',
 'себе',
 'себя',
 'сейчас',
 'со',
 'совсем',
 'так

In [59]:
def get_sentence_avg_vector(lemmas, model):
    vectors = []
    for word in lemmas:
        if word in russian_stopwords:
            continue
        if word in model.wv:
            vectors.append(model.wv[word])
    if not vectors:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

def get_sentence_weighted_vector(lemmas, tf_dict, idf_dict, model):
    weighted = np.zeros(model.wv.vector_size)
    total_weight = 0.0
    for word in lemmas:
        if word in russian_stopwords:
            continue
        if word in model.wv:
            tf = tf_dict.get(word, 0)
            idf = idf_dict.get(word, 0)
            weight = tf * idf
            weighted += weight * model.wv[word]
            total_weight += weight
    if total_weight > 0:
        return weighted / total_weight
    return get_sentence_avg_vector(lemmas, model)   # fallback

In [16]:
doc_avg_vectors = []
doc_weighted_vectors = []

for i, lemmas in enumerate(all_lemma_sentences):
    vec_avg = get_sentence_avg_vector(lemmas, model)
    doc_avg_vectors.append(vec_avg)
    
    vec_weighted = get_sentence_weighted_vector(lemmas, doc_tfs[i], idf, model)
    doc_weighted_vectors.append(vec_weighted)


In [17]:
def search_embedding(query_text, doc_vectors, sentences, model, mode='avg'):
    query_lemmas = preprocess_text(query_text)
    query_tf = Counter(query_lemmas)
    
    if mode == 'avg':
        query_vec = get_sentence_avg_vector(query_lemmas, model)
    else:
        query_vec = get_sentence_weighted_vector(query_lemmas, query_tf, idf, model)
    
    norm_q = np.linalg.norm(query_vec)
    
    results = []
    for i, doc_vec in enumerate(doc_vectors):
        norm_d = np.linalg.norm(doc_vec)
        sim = np.dot(query_vec, doc_vec) / (norm_q * norm_d)
        results.append((sim, sentences[i]))
    
    results.sort(key=lambda x: x[0], reverse=True)
    return results[:10]

In [60]:
queries = [
    "Турецкий футболист успешно выступал и в лёгкой атлетике.",
    "Коктейль назвали в честь Нью-Йорка, хотя, вероятно, создан он был в Чикаго.",
    "В гроте карстовой пещеры инженеры-студенты летали на воздушном шаре, а инженеры-профессионалы — запустили электростанцию."
]

for q_idx, query_text in enumerate(queries, 1):
    print(f"\nЗапрос {q_idx}: {query_text}")
    
    res_avg = search_embedding(query_text, doc_avg_vectors, all_sentences, model, mode='avg')
    print("\n--- FastText + усреднение (косинусная мера) ---")
    for score, sent in res_avg:
        print(f"{score:.6f}\t{sent}")
    
    res_weighted = search_embedding(query_text, doc_weighted_vectors, all_sentences, model, mode='weighted')
    print("\n--- FastText + tf-idf взвешивание (косинусная мера) ---")
    for score, sent in res_weighted:
        print(f"{score:.6f}\t{sent}")


Запрос 1: Турецкий футболист успешно выступал и в лёгкой атлетике.

--- FastText + усреднение (косинусная мера) ---
0.743058	В лёгкой атлетике неприменим опыт тяжёлой атлетики , где ввели новую сетку весовых категорий и тем самым попросту отменили все предыдущие мировые рекорды.
0.741223	Правила Победителем соревнований в лёгкой атлетике становится атлет или команда, которые показали наилучший результат в финальном забеге или в финальных попытках технических дисциплин в рамках правил.
0.738578	Значительное количество мировых рекордов в лёгкой атлетике вызывает законные подозрения специалистов, хотя атлеты не были пойманы и сами не признались.
0.730314	Однако и в 21 веке, наряду с тяжёлой атлетикой, велоспортом, плаванием — лёгкая атлетика остаётся видом спорта, наиболее подверженным проблеме допинга.
0.727622	Лёгкая атлетика Материал из Википедии — свободной энциклопедии Лёгкая атле́тика — олимпийский вид спорта.
0.723886	После объединения Германии особенно большое число пойманных и д

**Домашняя работа №10**

In [50]:
ground_truth_text = {
    0: {  # запрос 1
        2: [ 
            "Мелих Котанджа 1915 — 8 июня 1986, Стамбул) — турецкий футболист, играл на позиции нападающего.",
            "Он выступал в лёгкой атлетике, специализируясь на беге на различные дистанции, пятиборье и многоборье."
        ],
        1: [] 
    },
    1: {  # запрос 2
        2: [
            "Предполагается, что Нью-Йорк сауэр был Год создания 1880-е изобретён в 1880-е годы в Чикаго как вариация Приготовление и подача виски сауэра, впервые упомянутого в 1862 году"
        ],
        1: [
            "[1] Коктейль похож на виски сауэр , отличается от него добавлением красного вина.",
            "В астрономии В честь Чикаго назван астероид (334) Чикаго, открытый в 1892 году."
        ]
    },
    2: {  # запрос 3
        2: [
            "Полёт на воздушном шаре В декабре 2001 года студенты парижской Политехнической школы задумали уникальный проект: полёт на воздушном шаре под землёй.",
            "Запуск электростанции Проект Политехнической школы вновь пробудил интерес к Ла-Верне.",
            "Регион Новая Аквитания Район Атлантические Содержание Пиренеи История Первые экспедиции Открытие Ла-Верны Сооружение туннеля Полёт на воздушном шаре Запуск электростанции",
            "Ла-Верна Материал из Википедии — свободной энциклопедии Зал Ла-Верна (фр. Grotte (Salle) de La Verna) — Ла-Верна гигантский грот (255 × 245 × 194 м) в карстовой фр. Salle La Verna (Grotte de La Verna) пещере Гуфр-де-Партаж (Gouffre des Partages), входящей в разветвлённую (около 2000 камер) пещерную систему Пьер-Сен-Мартен."
        ],
        1: [
            "Стартовав около 14:00 от замка ла Мюетт в Булонском лесу (парк на западной окраине Парижа), воздушный шар поднялся на высоту около 3000 футов (1 км).",
            "Название воздушный шар получил по фамилии изобретателей, братьев Монгольфье — Жозеф-Мишеля и Жак-Этьенна.",
            "По состоянию на 2020 год Пьер-Сен-Мартен, [1] насчитывающая около 2 тысяч гротов и имеющая нижнюю отметку в —1342 м , входит в топ-30 пещер мира по длине и топ-20 по глубине."
        ]
    }
}

In [19]:
def find_sentence_index(sentence_text, sentences):
    for i, s in enumerate(sentences):
        if sentence_text.strip() in s:
            return i
    return -1

In [51]:
ground_truth = {}
for q_idx, gt_dict in ground_truth_text.items():
    true_scores = {}
    for score, sentences_list in gt_dict.items():
        for sent_text in sentences_list:
            idx = find_sentence_index(sent_text, all_sentences)
            if idx != -1:
                true_scores[idx] = score
            else:
                print(f"Внимание: предложение для запроса {q_idx+1} не найдено:\n{sent_text[:100]}...")
    ground_truth[q_idx] = true_scores

In [52]:
import math

def dcg(relevance_scores, k=10):
    dcg_val = 0.0
    for i, rel in enumerate(relevance_scores[:k]):
        dcg_val += rel / math.log2(i + 2)
    return dcg_val

def ndcg(ranked_indices, true_scores, k=10):
    pred_scores = [true_scores.get(idx, 0) for idx in ranked_indices[:k]]
    dcg_val = dcg(pred_scores, k)
    ideal_scores = sorted(true_scores.values(), reverse=True)
    idcg_val = dcg(ideal_scores, k)
    if idcg_val == 0:
        return 0.0
    return dcg_val / idcg_val

In [61]:
norm_tf_docs = [compute_norm(tf) for tf in doc_tfs]
norm_tfidf_docs = [compute_norm(tf, idf) for tf in doc_tfs]

rankings = {
    'TF': [],
    'TF-IDF': [],
    'BM25': [],
    'FastText_avg': [],
    'FastText_weighted': []
}

for q_idx, query_text in enumerate(queries):
    query_lemmas = preprocess_text(query_text)
    query_tf = Counter(query_lemmas)
    norm_tf_q = compute_norm(query_tf)
    query_tfidf = {word: tf * idf.get(word, 0) for word, tf in query_tf.items()}
    norm_tfidf_q = compute_norm(query_tf, idf)

    # ---------- TF ----------
    scores_tf = []
    for i in range(N):
        sim = cosine_similarity(query_tf, doc_tfs[i], norm_tf_q, norm_tf_docs[i])
        scores_tf.append((sim, i))
    scores_tf.sort(key=lambda x: x[0], reverse=True)
    rankings['TF'].append([idx for score, idx in scores_tf[:10]])

    # ---------- TF-IDF ----------
    scores_tfidf = []
    for i in range(N):
        doc_tfidf = {word: tf * idf.get(word, 0) for word, tf in doc_tfs[i].items()}
        sim = cosine_similarity(query_tfidf, doc_tfidf, norm_tfidf_q, norm_tfidf_docs[i])
        scores_tfidf.append((sim, i))
    scores_tfidf.sort(key=lambda x: x[0], reverse=True)
    rankings['TF-IDF'].append([idx for score, idx in scores_tfidf[:10]])

    # ---------- BM25 ----------
    scores_bm25 = []
    for i in range(N):
        score = bm25_score(query_tf, doc_tfs[i], doc_lengths[i], avgdl, idf)
        scores_bm25.append((score, i))
    scores_bm25.sort(key=lambda x: x[0], reverse=True)
    rankings['BM25'].append([idx for score, idx in scores_bm25[:10]])

    # ---------- FastText----------
    q_avg = get_sentence_avg_vector(query_lemmas, model)
    norm_q = np.linalg.norm(q_avg)
    scores_ft_avg = []
    for i, dvec in enumerate(doc_avg_vectors):
        norm_d = np.linalg.norm(dvec)
        sim = np.dot(q_avg, dvec) / (norm_q * norm_d) if norm_q > 0 and norm_d > 0 else 0.0
        scores_ft_avg.append((sim, i))
    scores_ft_avg.sort(key=lambda x: x[0], reverse=True)
    rankings['FastText_avg'].append([idx for score, idx in scores_ft_avg[:10]])

    # ---------- FastText (взвешенный) ----------
    q_weighted = get_sentence_weighted_vector(query_lemmas, query_tf, idf, model)
    norm_qw = np.linalg.norm(q_weighted)
    scores_ft_w = []
    for i, dvec in enumerate(doc_weighted_vectors):
        norm_d = np.linalg.norm(dvec)
        sim = np.dot(q_weighted, dvec) / (norm_qw * norm_d) if norm_qw > 0 and norm_d > 0 else 0.0
        scores_ft_w.append((sim, i))
    scores_ft_w.sort(key=lambda x: x[0], reverse=True)
    rankings['FastText_weighted'].append([idx for score, idx in scores_ft_w[:10]])

In [66]:
ndcg_results = {s_model: [] for s_model in rankings}
for q_idx in range(len(queries)):
    for s_model in rankings:
        ndcg_val = ndcg(rankings[s_model][q_idx], ground_truth[q_idx], k=10)
        ndcg_results[s_model].append(ndcg_val)

print("РЕЗУЛЬТАТЫ NDCG@10 ПО МОДЕЛЯМ И ЗАПРОСАМ")
print("\nЗапросы:")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")
print()

print("{:<20} {:>12} {:>12} {:>12} {:>12}".format("Модель", "Запрос 1", "Запрос 2", "Запрос 3", "Среднее"))

for model in ndcg_results:
    scores = ndcg_results[model]
    avg = sum(scores) / len(scores)
    print("{:<20} {:>12.4f} {:>12.4f} {:>12.4f} {:>12.4f}".format(model, scores[0], scores[1], scores[2], avg))

РЕЗУЛЬТАТЫ NDCG@10 ПО МОДЕЛЯМ И ЗАПРОСАМ

Запросы:
  1. Турецкий футболист успешно выступал и в лёгкой атлетике.
  2. Коктейль назвали в честь Нью-Йорка, хотя, вероятно, создан он был в Чикаго.
  3. В гроте карстовой пещеры инженеры-студенты летали на воздушном шаре, а инженеры-профессионалы — запустили электростанцию.

Модель                   Запрос 1     Запрос 2     Запрос 3      Среднее
TF                         0.3869       0.3194       0.3764       0.3609
TF-IDF                     1.0000       0.6347       0.8345       0.8230
BM25                       0.9197       0.6534       0.8656       0.8129
FastText_avg               0.0000       0.5117       0.3542       0.2886
FastText_weighted          0.6131       0.2015       0.2035       0.3394


In [67]:
print("ИДЕАЛЬНАЯ РАЗМЕТКА (GROUND TRUTH)")
for q_idx, q_text in enumerate(queries):
    print(f"\nЗапрос {q_idx+1}: {q_text}")
    true_dict = ground_truth[q_idx]
    if not true_dict:
        print("  Нет релевантных предложений")
    else:
        # сортируем по убыванию оценки
        for idx, score in sorted(true_dict.items(), key=lambda x: x[1], reverse=True):
            print(f"  [Оценка {score}] {all_sentences[idx][:200]}...")

print()
print()
print("ПРИМЕР ВЫДАЧИ ДЛЯ ЛУЧШЕЙ МОДЕЛИ (TF-IDF) – ТОП-10")
for q_idx, q_text in enumerate(queries):
    print(f"\nЗапрос {q_idx+1}: {q_text}")
    for rank, idx in enumerate(rankings['TF-IDF'][q_idx][:10], 1):
        rel = ground_truth[q_idx].get(idx, 0)
        print(f"  {rank}. [релевантность={rel}] {all_sentences[idx][:150]}...")

ИДЕАЛЬНАЯ РАЗМЕТКА (GROUND TRUTH)

Запрос 1: Турецкий футболист успешно выступал и в лёгкой атлетике.
  [Оценка 2] Мелих Котанджа 1915 — 8 июня 1986, Стамбул) — турецкий футболист, играл на позиции нападающего....
  [Оценка 2] Он выступал в лёгкой атлетике, специализируясь на беге на различные дистанции, пятиборье и многоборье....

Запрос 2: Коктейль назвали в честь Нью-Йорка, хотя, вероятно, создан он был в Чикаго.
  [Оценка 2] Содержание История Состав Приготовление См. также Коктейль «Нью-Йорк сауэр» в бокале олд Примечания фэшен Происхождение и тип коктейля История Тип сауэр Место создания Чикаго Предполагается, что Нью-Й...
  [Оценка 1] [1] Коктейль похож на виски сауэр , отличается от него добавлением красного вина....
  [Оценка 1] В астрономии В честь Чикаго назван астероид (334) Чикаго, открытый в 1892 году....

Запрос 3: В гроте карстовой пещеры инженеры-студенты летали на воздушном шаре, а инженеры-профессионалы — запустили электростанцию.
  [Оценка 2] Полёт на воздушном шаре

**Домашняя работа №11**

In [71]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

def get_hf_embeddings(texts, tokenizer, m):
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    m = m.to(device)
    with torch.no_grad():
        model_output = m(**encoded_input)
    return model_output.pooler_output.cpu().numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---------- 1. E5 Small ----------
e5_small_tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-small')
e5_small_model = AutoModel.from_pretrained('intfloat/multilingual-e5-small')
collection_e5_small = get_hf_embeddings(all_sentences, e5_small_tokenizer, e5_small_model)
query_e5_small = get_hf_embeddings(queries, e5_small_tokenizer, e5_small_model)

# ---------- 2. E5 Base ----------
e5_base_tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
e5_base_model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')
collection_e5_base = get_hf_embeddings(all_sentences, e5_base_tokenizer, e5_base_model)
query_e5_base = get_hf_embeddings(queries, e5_base_tokenizer, e5_base_model)

# ---------- 3. LaBSE (en-ru) ----------
labse_tokenizer = AutoTokenizer.from_pretrained('cointegrated/LaBSE-en-ru')
labse_model = AutoModel.from_pretrained('cointegrated/LaBSE-en-ru')
collection_labse = get_hf_embeddings(all_sentences, labse_tokenizer, labse_model)
query_labse = get_hf_embeddings(queries, labse_tokenizer, labse_model)

print("Все эмбеддинги успешно вычислены.")

Using device: cpu


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

'timed out' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/sentencepiece.bpe.model
Retrying in 1s [Retry 1/5].
'timed out' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/sentencepiece.bpe.model
Retrying in 2s [Retry 2/5].
'timed out' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/sentencepiece.bpe.model
Retrying in 4s [Retry 3/5].
'timed out' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/sentencepiece.bpe.model
Retrying in 8s [Retry 4/5].
'timed out' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/sentencepiece.bpe.model
Retrying in 8s [Retry 5/5].
'timed out' thrown while requesting HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/sentencepiece.bpe.model


ImportError: 
 requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_ndcg(query_emb, doc_emb, ground_truth, k=10):
    ndcg_scores = {}
    num_queries = query_emb.shape[0]
    for q_idx in range(num_queries):
        sim = cosine_similarity(query_emb[q_idx:q_idx+1], doc_emb)[0]
        ranked_indices = np.argsort(sim)[::-1][:k]
        true_scores = ground_truth.get(q_idx, {})
        ndcg_val = ndcg(ranked_indices.tolist(), true_scores, k=k)
        ndcg_scores[q_idx] = ndcg_val
    return ndcg_scores

ndcg_e5_small = evaluate_ndcg(query_e5_small, collection_e5_small, ground_truth, k=10)
ndcg_e5_base = evaluate_ndcg(query_e5_base, collection_e5_base, ground_truth, k=10)
ndcg_labse = evaluate_ndcg(query_labse, collection_labse, ground_truth, k=10)

all_ndcg = {
    'E5 Small': ndcg_e5_small,
    'E5 Base': ndcg_e5_base,
    'LaBSE': ndcg_labse
}

In [ ]:
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА NDCG@10 ДЛЯ ВСЕХ МОДЕЛЕЙ")
print("="*80)
print("\nЗапросы:")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")
print()

# Формируем строку заголовка
header = "{:<20} {:>12} {:>12} {:>12} {:>12}".format("Модель", "Запрос 1", "Запрос 2", "Запрос 3", "Среднее")
print(header)
print("-" * len(header))

# Выводим результаты для каждой модели
for model_name, scores_dict in all_ndcg.items():
    scores = [scores_dict[i] for i in range(3)]
    avg = sum(scores) / len(scores)
    print("{:<20} {:>12.4f} {:>12.4f} {:>12.4f} {:>12.4f}".format(model_name, scores[0], scores[1], scores[2], avg))